# Macro Higiênico

In [1]:
(define (square-f x)
  (* x x))

In [2]:
(square-f 10)

100

# `define-syntax`

## `define-rules`

In [2]:
(define-syntax square-m
  (syntax-rules ()
    [(square-m x)
     (* x x)]))

In [3]:
(square-m 10)

100

### Efeitos Indesejados

In [4]:
(begin (display "valor: ") 3)

valor: 

3

In [5]:
(* (begin (display "valor: ") 3) 2)

valor: 

6

## Tarefa

Escreva um exemplo que produza um efeito indesejado quando usado o square-m, mas não o produza quando usado o square-f.

In [28]:
(define (square-f x)
  (* x x))

(define-syntax square-m
  (syntax-rules ()
    [(square-m x)
     (* x x)]))


(display (square-f (begin (display "valor: ") 3)))
(newline)
(display (square-m (begin (display "valor: ") 3)))

valor: 9
valor: valor: 9

In [31]:
(define (square-f x)
  (* x x))

; macro consertado
(define-syntax square-m
  (syntax-rules ()
    [(square-m x)
     (let ([v x])
       (* v v))]))


(display (square-f (begin (display "valor: ") 3)))
(newline)
(display (square-m (begin (display "valor: ") 3)))

valor: 9
valor: 9

## Por que é Higiênico

In [44]:
; para rodar define-macro com o racket
(require mzlib/defmacro)
(require (for-syntax racket/base))

(define-macro (swap-n! a b)
  `(let ([temp ,a])
     (set! ,a ,b)
     (set! ,b temp)))

In [45]:
(let ([x 1]
      [y 2])
  (swap-n! x y)
  (list x y))

'(2 1)

In [46]:
(let ([temp 1]
      [y 2])
  (swap-n! temp y)
  (list temp y))

'(1 2)

#### Ecercício

In [ ]:
; raciocínio do porquê o exemplo de cima resulta em '(1 2):

(define-macro (swap-n! a b)
  `(let ([temp ,a])
     (set! ,a ,b)
     (set! ,b temp)))

; linha
(swap-n! temp y)
; --->
; (let) temp = 'temp
; temp = y (temp = 2)
; y = 'temp
; portanto vai dar:
; (list temp y) -> (list 1 temp) -> '(1 2)


(swap-n! temp y)
; --->
(let ([temp temp])
    (set! temp 2)  ; temp interno = 2
    (set! y temp)  ; y = temp interno (y = 2)
)

; aqui fora temp = 1  e  y = 2


In [48]:
(define-syntax swap-i!
  (syntax-rules ()
    [(swap-i! a b)
     (let ([temp a])
       (set! a b)
       (set! b temp))]))

In [49]:
(let ([x 1]
      [y 2])
  (swap-i! x y)
  (list x y))

'(2 1)

In [50]:
(let ([temp 1]
      [y 2])
  (swap-i! temp y)
  (list temp y))

'(2 1)

## `syntax-rules <literals>`

In [51]:
(define-syntax simple-path
  (syntax-rules (to)
    [(simple-path node1 to node2)
     '(node1 . node2)]))

In [52]:
(simple-path a to b)

'(a . b)

## Explorando o potencial dos símbolos: `->`

In [53]:
(define-syntax simple-path
  (syntax-rules (->)
    [(simple-path node1 -> node2)
     '(node1 . node2)]))

In [54]:
(simple-path a -> b)

'(a . b)

In [55]:
(define-syntax simple-path
  (syntax-rules (-> <-)
    [(simple-path node1 -> node2)
     '(node1 . node2)]
    [(simple-path node1 <- node2)
     '(node2 . node1)]))

In [56]:
(simple-path a -> b)

'(a . b)

In [57]:
(simple-path a <- b)

'(b . a)

In [58]:
(define-syntax simple-path
  (syntax-rules (-> <- <->)
    [(simple-path node1 -> node2)
     '(node1 . node2)]
    [(simple-path node1 <- node2)
     '(node2 . node1)]
    [(simple-path node1 <-> node2)
     '((node1 . node2) (node2 . node1))]))

In [59]:
(simple-path a <-> b)

'((a . b) (b . a))

## `syntax-case`

In [60]:
(define-syntax simple-path
  (lambda (stx)
    (syntax-case stx (-> <- <->)

      [(simple-path node1 -> node2)
       #'(quote (node1 . node2))])))

In [61]:
(simple-path a -> b)

'(a . b)

In [62]:
(define-syntax simple-path
  (lambda (stx)
    (syntax-case stx (-> <- <->)

      [(simple-path node1 -> node2)
       #'(quote (node1 . node2))]

      [(simple-path node1 <- node2)
       #'(quote (node2 . node1))]

      [(simple-path node1 <-> node2)
       #'(quote ((node1 . node2)
                 (node2 . node1)))])))

In [63]:
(simple-path a <-> b)

'((a . b) (b . a))

#### Exercício
Fazer syntax-rules usando o syntax-case (com um padrão só).

In [ ]:
; exemplo de syntax-rules
(define-syntax simple-path
  (syntax-rules (->)
    [(simple-path node1 -> node2)
     '(node1 . node2)]))

(define-syntax syntax-rules*
  (lambda (stx)
    (syntax-case stx ()

      ((syntax-rules* (k ...) ((keyword . pattern) template) ...)
        #'(lambda (x)
            (syntax-case x (k ...)
              ((keyword . pattern)
               #'template)
              ...
            )
          )
      )
    )
  )
)


## Além do pattern matching

In [64]:
(define-syntax simple-path
  (lambda (stx)
    (syntax-case stx (-> <- <->)

      [(simple-path node1 -> node2)
        (if (identifier? #'node2)
           #'(quote (node1 . node2))
           #'(quote ((node1 . $) ($ . node2))))])))

In [65]:
(simple-path a -> b)

'(a . b)

In [66]:
(simple-path a -> 3)

'((a . $) ($ . 3))

## `syntax->datum`

In [72]:
3

3

In [73]:
#'3

#<syntax:eval:1:2 3>

In [74]:
(syntax->datum #'3)

3

In [75]:
(define-syntax simple-path
  (lambda (stx)
    (syntax-case stx (-> <- <->)

      [(simple-path node1 -> node2)
        (if (number? (syntax->datum #'node2))
           #'(quote ((node1 . $) ($ . node2)))
           #'(quote (node1 . node2)))])))

In [76]:
(simple-path a -> 3)

'((a . $) ($ . 3))